# Deep Learning Clínico Avanzado: Transfer Learning (Modelos Preentrenados)

¡Subimos el nivel a la excelencia! Has pedido que usemos **Modelos Preentrenados**. Estas maravillas de la Inteligencia Artificial (como VGG16 o ResNet) fueron entrenadas durante meses en supercomputadoras de Google y Microsoft leyendo millones de imágenes para aprender a *"ver y distinguir bordes y patrones"* a la perfección.

**¿El problema en biomedicina?** Nuestro **Breast Cancer Wisconsin Dataset** es *Tabular* (30 resultados numéricos de laboratorio, no fotografías).

**¿La Solución Magistral? ¡TABULAR-TO-IMAGE!**
Vamos a usar un truco avanzado de Bioinformática: Convertiremos los 30 signos vitales y celulares de cada paciente en una «Matriz Fotográfica» o Código de Barras Óptico de 32x32 píxeles. 
De esta forma, engañaremos a 4 Modelos Visuales Preentrenados para que hagan un diagnóstico clínico sobre esta "imagen de sangre".

In [ ]:
# Instalamos la base clínica de ML Clásico
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler  # Usaremos MinMax para simular fotometría (0 a 1)
from imblearn.over_sampling import SMOTE

# Nuestros Titanés Preentrenados de Deep Learning (Transfer Learning)
import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2, DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

---  
## 1. Carga de Biopsias y Exploración Médica (EDA)

El paso inicial de siempre: Conocer los datos de nuestros pacientes oncológicos reales.

In [ ]:
cancer_data = load_breast_cancer()
df = pd.DataFrame(data=cancer_data.data, columns=cancer_data.feature_names)
df['Diagnostico'] = cancer_data.target # 0: Maligno, 1: Benigno

# Análisis Rápido de Correlación: Qué rasgos se ligan a la malignidad antes de ser fotos
plt.figure(figsize=(10, 4))
top_features = df.corr()['Diagnostico'].abs().sort_values(ascending=False).head(10).index
sns.heatmap(df[top_features].corr(), annot=True, cmap="coolwarm", fmt=".2f", cbar=False)
plt.title('EDA: Correlación Fisiológica de las 10 peores células tumorales')
plt.show()

---  
## 2. Preprocesamiento: Limpieza y Balanceo (Standard Care)

Separaremos variables, dividiremos urgencias (test) y aplicaremos `SMOTE` para que no haya desbalance entre tumores sanos y enfermos.

In [ ]:
X = df.drop(columns=['Diagnostico'])
y = df['Diagnostico']

# Split Reservado del 20% para calificar a la IA
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# Como convertiremos esto a píxeles RGB, los píxeles viven en el mundo de [0 a 1] matemáticamente.
# Usaremos MinMaxScaler para aplastar gentilmente todos los milimetros celulares a rangos de color 0 a 1.
scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scl = scaler.fit_transform(X_train_raw)
X_test_scl = scaler.transform(X_test_raw)

# SMOTE Sintético
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scl, y_train)

print("Balanceo Citológico Perfecto:")
print(pd.Series(y_train_bal).value_counts())

---  
## 3. LA MAGIA: Tabular-To-Image Transformation

Aquí sucede el evento central:
1. Cada paciente tiene `30` métricas.
2. Le agregaremos `6` ceros de relleno para tener `36`.
3. Convertimos `36` a un cuadrado perfecto de `6x6`.
4. Interpolaremos e inflaremos mágicamente ese cuadrado a una resolución visual de `32x32`.
5. Lo clonaremos en 3 capas de color fotográfico (`RGB`).

**!Voilà! Hemos mutado un examen de sangre numérico tabular en una "Radiografía Algorítmica"!**

In [ ]:
# 1 y 2. Añadir ceros al final para completar a 36
X_train_pad = np.pad(X_train_bal, ((0,0), (0,6)), mode='constant')
X_test_pad  = np.pad(X_test_scl, ((0,0), (0,6)), mode='constant')

# 3. Cambiar la forma a microscopio óptico 2D (6x6 de profundidad de 1 color)
X_train_2d = X_train_pad.reshape(-1, 6, 6, 1)
X_test_2d  = X_test_pad.reshape(-1, 6, 6, 1)

# 4 y 5. Ampliar (Bilinear Resize) a 32x32 y Convertir a RGB (3 Canales Ópticos)
X_train_img = tf.image.resize(X_train_2d, [32, 32])
X_train_img = tf.image.grayscale_to_rgb(X_train_img).numpy() # Shape: (N, 32, 32, 3)

X_test_img = tf.image.resize(X_test_2d, [32, 32])
X_test_img = tf.image.grayscale_to_rgb(X_test_img).numpy()

print(f"Nuevos tensores de pacientes biomédicos listos para Transfer Learning: {X_train_img.shape}")

plt.figure(figsize=(10,3))
for i in range(4):
    plt.subplot(1, 4, i+1)
    plt.imshow(np.clip(X_train_img[i], 0, 1), cmap='magma')
    estado = "Benigno" if y_train_bal.iloc[i] == 1 else "Maligno"
    plt.title(f"Patrón: {estado}", fontsize=10)
    plt.axis('off')
plt.suptitle('Radiografías Tabulares Sintéticas (El Paciente convertido a Píxeles RGB)', fontsize=14)
plt.show()

---  
## 4. Transfer Learning (4 Modelos Preentrenados Gigantes)

Llegó el momento. Descargaremos los "Cerebros" de las inteligencias preentrenadas más potentes construidas por Oxford, Google y Microsoft. 

Les **congelaremos** sus conocimientos aprendidos de la vida (`trainable = False`), y solo le agregaremos una "Capa en la cabeza" (`Dense(1)`) para que aprendan a relacionar su súper-visión con nuestro cuadro de Cáncer de Mama.

In [ ]:
input_layer_optico = Input(shape=(32, 32, 3))

def adaptar_modelo_preentrenado(base_model_func, model_name):
    # Importamos el cerebro sin la última capa (include_top=False)
    red_base_preentrenada = base_model_func(weights='imagenet', include_top=False, input_tensor=input_layer_optico)
    
    # ¡IMPORTANTE en Transfer Learning! Congelar (No sobreescribir billones de horas de cálculo previas)
    red_base_preentrenada.trainable = False
    
    # Le instalamos un conector para aterrizarlo a nuestro problema Oncológico (Binario)
    x = GlobalAveragePooling2D()(red_base_preentrenada.output)
    x = Dense(64, activation='relu')(x)       # Capa deductiva media
    x = tf.keras.layers.Dropout(0.2)(x)       # Prevención Anti-Efecto de red
    salida = Dense(1, activation='sigmoid')(x)  # Cáncer(0) o Sano(1)
    
    modelo_final = Model(inputs=red_base_preentrenada.input, outputs=salida, name=model_name)
    modelo_final.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    return modelo_final

# Nuestros 4 Expertos Universales
consorcio_ia = {
    'VGG16 (Universidad de Oxford)': adaptar_modelo_preentrenado(VGG16, 'vgg_medical'),
    'ResNet50 (Microsoft)': adaptar_modelo_preentrenado(ResNet50, 'resnet_medical'),
    'MobileNetV2 (Google)': adaptar_modelo_preentrenado(MobileNetV2, 'mobile_medical'),
    'DenseNet121 (Huang et al)': adaptar_modelo_preentrenado(DenseNet121, 'dense_medical')
}

pred_test = {}
prob_test = {}

print("===== ENTRENAMIENTO DE TRANSFER LEARNING (Adaptación Científica) =====")
for nombre, modelo_ia in consorcio_ia.items():
    print(f"Inyectando Patrones Clínicos en el lóbulo frontal de: {nombre}...")
    # 15 Vueltas de enseñanza en la capa cabezal
    modelo_ia.fit(X_train_img, y_train_bal, epochs=15, batch_size=32, verbose=0) 
    
    probabilidades = modelo_ia.predict(X_test_img, verbose=0).flatten()
    prob_test[nombre] = probabilidades
    pred_test[nombre] = (probabilidades > 0.5).astype(int)

print("\n¡Todos los modelos masivos preentrenados han sido aclimatados a Biomedicina y han rendido el examen!")

---  
## 5. Auditoría de Múltiples Redes (Resultados Oncológicos)

¿Qué Super-Red "vio" mejor los tumores escondidos en nuestros simuladores de tabulares?
Revisemos las Matrices de Confusión.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for idx, (nombre, predicciones) in enumerate(pred_test.items()):
    # Evaluador de Aciertos
    cm = confusion_matrix(y_test, predicciones)
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', 
                xticklabels=['Maligno (0)', 'Benigno (1)'], 
                yticklabels=['Maligno (0)', 'Benigno (1)'], ax=axes[idx])
    
    axes[idx].set_title(f"Transfer Learning:\n{nombre}", fontweight='bold')
    axes[idx].set_ylabel('Laboratorio Real')
    axes[idx].set_xlabel('Preentrenado AI')

plt.tight_layout()
plt.show()

### 5.1 Calidad Clínica (Classification Reports)

¡Observa tu Recall en Malignos! Es posible que, dada la altísima complejidad de las redes preentrenadas sobre una imagen tan burda numéricamente construida, algunas redes fallen espectacularmente (las tan grandes ResNet se despistan) mientras que otras como DenseNet extraen oro fotográfico.

In [ ]:
for nombre, predicciones in pred_test.items():
    print(f"\n>>> {nombre} <<<")
    print(classification_report(y_test, predicciones, target_names=['Maligno (0)', 'Benigno (1)']))

### 5.2 Curvas ROC Multi-Red y Final

El dictamen conclusivo sobre qué Red Preentrenada es la mejor adaptada al problema de Tabular-to-Image se juzga debajo.

In [ ]:
plt.figure(figsize=(10, 8))

for nombre, probs in prob_test.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f"{nombre} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Adivinación (50%)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa Falsos Positivos (Gente asustada por Falso Cáncer)')
plt.ylabel('Tasa Verdaderos Positivos (Diagnósticos Salvadores)')
plt.title('Batalla Preentrenada: La mejor IA para imágenes biomédicas raras')
plt.legend(loc="lower right")
plt.grid(alpha=0.2)
plt.show()

## Resumen del Doctor de Datos

Has desbloqueado uno de los sistemas más avanzados empíricamente en biología computacional: **Utilizar Transfer Learning (Modelos pesados de Deep Learning Visual) aplicado en Data puramente Numérica.**

1. Convertimos tus tabulaciones médicas (`X`) a **matriz de colores RGB 32x32**.
2. Descargamos las plantillas de `VGG16`, `ResNet`, y `MobileNet` sin requerir tarjetas gráficas monstruosas (porque congelamos sus pesos y solo entrenamos durante minutos nuestro dictamen final). 
3. Probamos en Urgencias cómo un cerebro que nació mirando Perros, Autos y Montañas en *ImageNet*, **fue exitosamente hackeado** y reconectado analíticamente para examinar biopsias y emitir predicciones de Cáncer de Mama real.

¡Eres oficialmente imparable! ¡No hay data que no puedas pasar por una red Neuronal usando este método de interconexión dimensional Tabular-to-Image!

---
## 6. CNN Básica para Imágenes

Antes de saltar a modelos gigantes, es fundamental entender cómo es una CNN (Red Neuronal Convolucional) básica desde cero. 
A continuación crearemos una arquitectura sencilla con capas `Conv2D` y `MaxPooling2D`.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_basic_cnn(input_shape=(64, 64, 3), num_classes=2):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(pool_size=(2, 2)),
        
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax' if num_classes > 2 else 'sigmoid')
    ])
    
    model.compile(\n        optimizer='adam', \n        loss='sparse_categorical_crossentropy' if num_classes > 2 else 'binary_crossentropy', \n        metrics=['accuracy']\n    )
    return model

basic_cnn = build_basic_cnn()
basic_cnn.summary()

---
## 7. Clasificación de Tumores Cerebrales (MRI)

Ahora subimos el nivel aplicándolo a un problema médico clásico: Clasificar Imágenes de Resonancia Magnética (MRI) del cerebro. 
Para esto podemos usar Arquitecturas Preentrenadas (o CNNs Profundas) adaptadas para detectar la presencia de tumores. Aquí montamos la cabecera de la red sobre una base ResNet50.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Input

def build_tumor_classifier(input_shape=(224, 224, 3)):
    # Cargamos ResNet50 sin la capa final y congelamos pesos
    base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=Input(shape=input_shape))
    base_model.trainable = False
    
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    # Salida binaria: Positivo a Tumor (1) o Sano (0)
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=base_model.input, outputs=outputs, name='Brain_Tumor_Classifier')
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

tumor_classifier = build_tumor_classifier()
tumor_classifier.summary()

---
## 8. Segmentación Médica de Tumores (U-Net)

En cirugías y radioterapia no basta con saber *SI* hay un tumor, se necesita saber *DÓNDE* está exactamente. 
Para esto se usa la famosísima arquitectura **U-Net** (creada para biomedicina), la cual toma una imagen de un tamaño y produce una Máscara del mismo tamaño, coloreando exclusivamente la ubicación del tumor píxel por píxel.

In [ ]:
from tensorflow.keras.layers import UpSampling2D, concatenate

def build_unet(input_shape=(128, 128, 1)):
    inputs = Input(shape=input_shape)
    
    # --- ENCODER (Camino de Contracción) ---
    c1 = Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    p1 = MaxPooling2D((2, 2))(c1)
    
    c2 = Conv2D(32, (3, 3), activation='relu', padding='same')(p1)
    p2 = MaxPooling2D((2, 2))(c2)
    
    # --- CUELLO DE BOTELLA ---
    c3 = Conv2D(64, (3, 3), activation='relu', padding='same')(p2)
    
    # --- DECODER (Camino de Expansión) ---
    u4 = UpSampling2D((2, 2))(c3)
    u4 = concatenate([u4, c2])
    c4 = Conv2D(32, (3, 3), activation='relu', padding='same')(u4)
    
    u5 = UpSampling2D((2, 2))(c4)
    u5 = concatenate([u5, c1])
    c5 = Conv2D(16, (3, 3), activation='relu', padding='same')(u5)
    
    # --- SALIDA ---
    # 1 pixel de salida por cada pixel de entrada (Probabilidad de que ese pixel sea Tumor)
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c5)
    
    model = Model(inputs=[inputs], outputs=[outputs], name='UNet_Tumor_Segmentation')
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

unet_model = build_unet()
unet_model.summary()